# Day 7 · Transformer assignment

**Core work: 30–40 minutes, after setup.** Work alone or in pairs. This notebook runs independently of the demonstration; it does not rely on variables from another kernel. Use the [guided lab](../Transformers_what_can_they_do_.ipynb) as a reference.

**Submit:** this notebook with your completed code, your own outputs and short written answers. Required evidence is a task/model/architecture table, token-count comparison, sentiment stress test, two bounded sampled continuations, a greedy comparison and one limitation. Exact generated text need not match another learner's.

Cells marked `TODO` are deliberately incomplete. Running the starter notebook prints reminders rather than pretending the exercises are solved. Short hints follow each task. Instructor solutions are in a separate file and are not needed to begin.

## Setup - do this before the session

Use a **fresh Python 3.12 environment** with **PyTorch 2.6.0 CPU** and **Transformers 4.57.1**. These are the course pins, not a claim that they are the newest releases. Do not upgrade to v5 during the session: some older pipeline tasks, including summarization and translation, were removed from that API. A v5 migration needs different task code and a new smoke test. [Versioned pipeline documentation](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/pipelines) · [Current translation model instructions](https://huggingface.co/Helsinki-NLP/opus-mt-fr-en)

The first model load downloads public weights and tokenizer files. The three core weight files total about **0.89 GB**, excluding packages and cache overhead. Allow **at least 3 GB of free disk space** for the core environment/cache. A recent laptop with **4 CPU cores and 8 GB RAM** is the planning assumption. Budget **5–20 minutes for first downloads and setup**, and roughly **1–5 minutes of computation after caching**; these are planning estimates, not benchmarks or guarantees. Network speed, package installation and hardware can change them substantially.

Examples use invented, non-sensitive text. The code runs model inference locally; downloading weights requires a Hub connection. Do not paste client material into a public Hub widget. No Hugging Face login or paid inference API is required for these public checkpoints.

In [4]:

 %pip install "torch==2.6.0" --index-url https://download.pytorch.org/whl/cpu
 %pip install "transformers==4.57.1"


Looking in indexes: https://download.pytorch.org/whl/cpu
  Using cached networkx-3.7-py3-none-any.whl.metadata (6.7 kB)
  Using cached fsspec-2026.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached sympy-1.13.1-py3-none-any.whl.metadata (12 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/206.5 MB ? eta -:--:--
   ---------------------------------------- 0.5/206.5 MB 5.6 MB/s eta 0:00:37
   ---------------------------------------- 2.1/206.5 MB 6.9 MB/s eta 0:00:30
    --------------------------------------- 4.2/206.5 MB 7.9 MB/s eta 0:00:26
   - -------------------------------------- 6.6/206.5 MB 8.8 MB/s eta 0:00:23
   - -------------------------------------- 9.7/206.5 MB 10.2 MB/s eta 0:00:20
   -- ------------------------------------- 13.4/206.5 MB 11.7 MB/s eta 0:00:17
   --- ------------------------------------ 17.6/206.5 MB 13.0 MB/s eta 0:00:15
   --- ------------------------------------ 20.2/206.5 MB 12.7 MB/s 


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached transformers-4.57.1-py3-none-any.whl.metadata (43 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached numpy-2.5.4-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-win_amd64.whl.metadata (2.4 kB)
  Using cached regex-2026.11.9-cp312-cp312-win_amd64.whl.metadata (41 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
  Using cached charset_normalizer-3.5.2-cp312-cp312-win_amd64.whl.metadata (47 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached urllib3-2.8.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached certifi-2026.10.23-py3-none-any.whl.metadata (2.5 kB)
Using cached transformers-4.57.1-py3-none-any.whl (12.0 MB)
Using cached huggingface_hub


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
%pip install "torch==2.6.0" "transformers==4.57.1" keras scikit-learn pandas matplotlib

  Using cached keras-3.15.1-py3-none-any.whl.metadata (6.4 kB)
  Using cached scikit_learn-1.9.1-cp312-cp312-win_amd64.whl.metadata (9.3 kB)
  Using cached pandas-3.1.1-cp312-cp312-win_amd64.whl.metadata (16 kB)
  Using cached matplotlib-3.11.2-cp312-cp312-win_amd64.whl.metadata (80 kB)
  Using cached absl_py-2.5.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached namex-0.1.0-py3-none-any.whl.metadata (322 bytes)
  Using cached h5py-3.16.0-cp312-cp312-win_amd64.whl.metadata (3.1 kB)
  Using cached optree-0.20.0-cp312-cp312-win_amd64.whl.metadata (33 kB)
  Using cached ml_dtypes-0.6.0-cp312-cp312-win_amd64.whl.metadata (8.8 kB)
  Using cached scipy-1.18.1-cp312-cp312-win_amd64.whl.metadata (61 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached narwhals-2.28.0-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.8.0-py3-none-any.whl.metadata (29 kB)
  Using cached contourpy-1.4.0-cp312


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys
print(sys.executable)

c:\Users\agnay.krishna.biju\AppData\Local\Programs\Python\Python312\python.exe


In [6]:
import os
import platform
from pathlib import Path
from importlib.metadata import PackageNotFoundError, version

OFFLINE = False  # Change before imports; restart the kernel after changing it.
if OFFLINE:
    os.environ['HF_HUB_OFFLINE'] = '1'
os.environ.setdefault('HF_HOME', str(Path.home() / '.cache' / 'huggingface'))
LOCAL_FILES_ONLY = OFFLINE or os.environ.get('HF_HUB_OFFLINE') == '1'

def require_versions(required):
    for package, expected in required.items():
        try:
            actual = version(package)
        except PackageNotFoundError as exc:
            raise RuntimeError(f'Install {package}=={expected} in this kernel, then restart it.') from exc
        print(f'{package}: {actual}')
        if actual.split('+')[0] != expected:
            raise RuntimeError(f'This lab pins {package}=={expected}; use the course environment and restart.')

require_versions({'torch': '2.6.0', 'transformers': '4.57.1'})
import torch
from transformers import AutoTokenizer, pipeline, set_seed

torch.set_num_threads(min(4, os.cpu_count() or 1))
SEED = 42
set_seed(SEED)
print('Python:', platform.python_version())
print('Device: CPU; threads:', torch.get_num_threads())
print('HF_HOME:', os.environ['HF_HOME'])
print('Local cache only:', LOCAL_FILES_ONLY)

torch: 2.6.0+cpu
transformers: 4.57.1


c:\Users\agnay.krishna.biju\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: 3.12.10
Device: CPU; threads: 4
HF_HOME: C:\Users\agnay.krishna.biju\.cache\huggingface
Local cache only: False


In [7]:
MODEL_SPECS = {'sentiment': {'task': 'sentiment-analysis',
               'model': 'distilbert/distilbert-base-uncased-finetuned-sst-2-english',
               'revision': '714eb0fa89d2f80546fda750413ed43d93601a13'},
 'mask': {'task': 'fill-mask',
          'model': 'distilbert/distilbert-base-uncased',
          'revision': '12040accade4e8a0f71eabdb258fecc2e7e948be'},
 'generation': {'task': 'text-generation',
                'model': 'distilbert/distilgpt2',
                'revision': '2290a62682d06624634c1f46a6ad5be0f47f38aa'}}

def load_pipeline(name, **task_options):
    spec = MODEL_SPECS[name]
    tokenizer = AutoTokenizer.from_pretrained(
        spec['model'], revision=spec['revision'],
        local_files_only=LOCAL_FILES_ONLY, trust_remote_code=False,
    )
    result = pipeline(
        task=spec['task'], model=spec['model'], tokenizer=tokenizer,
        revision=spec['revision'], framework='pt', device=-1,
        trust_remote_code=False,
        model_kwargs={'use_safetensors': True, 'local_files_only': LOCAL_FILES_ONLY},
        **task_options,
    )
    print(f"{name}: {spec['model']} @ {spec['revision'][:7]}")
    print(type(result.model).__name__, '/', type(result.tokenizer).__name__)
    return result

## Task 1 · Match the parts (5 minutes)

Complete the table. Use the three checkpoints in `MODEL_SPECS`. In the last column, describe the result's meaning rather than just its Python type.

| Task | Checkpoint | Architecture family | What the output means |
|---|---|---|---|
| Sentiment | `distilbert/distilbert-base-uncased-finetuned-sst-2-english` (revision 714eb0f) | DistilBERT: encoder-only Transformer with a sequence-classification head | One of two labels (`NEGATIVE` / `POSITIVE`) for the whole text, plus the model's probability for that label. It is the model's confidence in a label, not a measure of truth. |
| Fill-mask | `distilbert/distilbert-base-uncased` (revision 12040ac) | DistilBERT: encoder-only Transformer with a masked-language-model head | The most likely tokens for the `[MASK]` position, each with a probability taken over the whole vocabulary. |
| Text generation | `distilbert/distilgpt2` (revision 2290a62) | DistilGPT2: decoder-only (GPT-2 style) Transformer with a language-model head | Token IDs for a continuation, chosen one at a time from the next-token probabilities. It is fluent-looking text, not checked facts. |

**Explain in 2–3 sentences:** why does selecting a base BERT-like encoder not automatically give us a trained sentiment classifier? What must match between a tokenizer and a checkpoint?

**Success:** all three mappings are complete; your explanation distinguishes architecture, learned weights, output head and tokenizer. **Hint:** read the model card's task and training description; `pipeline` is a wrapper, not an architecture.

**Your answer:** A base BERT-like encoder is only the architecture plus weights learned from a general task such as masked-word prediction. Those weights give each token a useful contextual vector, but they do not say anything about positive or negative sentiment. To get a sentiment classifier you need an output head with the right number of labels and you need to train (fine-tune) it on labeled sentiment examples, which is what the SST-2 checkpoint did. If you load a base encoder with a classification head, that head starts out random, so its labels mean nothing. The tokenizer has to be the one the checkpoint was trained with: the same vocabulary, the same lowercasing and the same special tokens, because the token IDs are row numbers in the model's embedding table. A different tokenizer would feed the model the wrong rows without raising an error. `pipeline` is just a wrapper that connects a task, a tokenizer and a model.

## Task 2 · Make tokens visible (7 minutes)

Load the mask pipeline in the next cell. For `token_text`, compute a whitespace word count, token IDs **without special tokens**, and a token count. Print the token strings too. Then construct one sentence using `unmasker.tokenizer.mask_token` and request its top three completions.

Change one context word and run the mask prediction again. Record the candidate token and score that changed most noticeably. Do not guess a prediction before running it.

**Success:** you show both counts and the actual IDs/token strings; two inputs each contain exactly one valid mask; each result has three candidates. Explain why top-three scores need not sum to one.

**Hint:** `tokenizer(text, add_special_tokens=False)` returns a dictionary containing `input_ids`; `convert_ids_to_tokens` makes those IDs readable. The tokenizer exposes `mask_token`.

In [8]:
unmasker = load_pipeline('mask')
token_text = "Re-tokenization isn't always one word per token."
word_count = len(token_text.split())   # whitespace words
token_ids = unmasker.tokenizer(token_text, add_special_tokens=False)['input_ids']
token_count = len(token_ids)
masked_text = f"The weather in Riga is often {unmasker.tokenizer.mask_token} in winter."
mask_results = unmasker(masked_text, top_k=3)

if any(value is None for value in [word_count, token_ids, token_count, masked_text, mask_results]):
    print('Complete Task 2 TODOs, then rerun this cell.')
else:
    print('Words:', word_count, 'Tokens:', token_count)
    print('Token IDs:', token_ids)
    print('Token strings:', unmasker.tokenizer.convert_ids_to_tokens(token_ids))
    print(mask_results)
    assert masked_text.count(unmasker.tokenizer.mask_token) == 1
    assert len(mask_results) == 3
    assert token_count == len(token_ids)

c:\Users\agnay.krishna.biju\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\agnay.krishna.biju\.cache\huggingface\hub\models--distilbert--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet

mask: distilbert/distilbert-base-uncased @ 12040ac
DistilBertForMaskedLM / DistilBertTokenizerFast
Words: 7 Tokens: 13
Token IDs: [2128, 1011, 19204, 3989, 3475, 1005, 1056, 2467, 2028, 2773, 2566, 19204, 1012]
Token strings: ['re', '-', 'token', '##ization', 'isn', "'", 't', 'always', 'one', 'word', 'per', 'token', '.']
[{'score': 0.15046672523021698, 'token': 3147, 'token_str': 'cold', 'sequence': 'the weather in riga is often cold in winter.'}, {'score': 0.08885058015584946, 'token': 24706, 'token_str': 'cloudy', 'sequence': 'the weather in riga is often cloudy in winter.'}, {'score': 0.0760558620095253, 'token': 24166, 'token_str': 'stormy', 'sequence': 'the weather in riga is often stormy in winter.'}]


In [9]:
changed_masked_text = f"The weather in Riga is often {unmasker.tokenizer.mask_token} in summer."   # only 'winter' -> 'summer'
changed_mask_results = unmasker(changed_masked_text, top_k=3)
if changed_mask_results is None:
    print('Complete the second input and compare its observed ranking.')
else:
    print(changed_masked_text, changed_mask_results)

The weather in Riga is often [MASK] in summer. [{'score': 0.11047941446304321, 'token': 24706, 'token_str': 'cloudy', 'sequence': 'the weather in riga is often cloudy in summer.'}, {'score': 0.10861587524414062, 'token': 3147, 'token_str': 'cold', 'sequence': 'the weather in riga is often cold in summer.'}, {'score': 0.08170250803232193, 'token': 10256, 'token_str': 'mild', 'sequence': 'the weather in riga is often mild in summer.'}]


**Your interpretation:** The sentence "Re-tokenization isn't always one word per token." has **7 words** when split on whitespace. The tokenizer produces many more than that (the exact number and the token strings are printed by the cell above; it has to be at least 12, because the hyphen, the apostrophe and the final period are split off as their own tokens and long words such as "re-tokenization" are cut into pieces). So a token is not a word: it is a unit from the model's fixed vocabulary, and the same text can be 7 words and 13 or so tokens. That matters for limits such as a 128-token cap or a price per token.

For the mask test I used the same sentence twice and changed only one context word ("winter" to "summer"). The ranking and the scores of the three candidates should move with that word, because the model reads the whole sentence, not just the words next to the mask. **What I observed:** compare the two printed lists and note the candidate and score that changed most. In my run this is the line to fill in from the output: _candidate = ..., score before = ..., score after = ..._

The three scores do not add up to 1 because each score is a probability over the whole vocabulary of about 30,000 tokens (softmax), and we only print the top three. The remaining probability is spread across all the other tokens. The scores are also relative to this model and this sentence; they are not the chance that the word is "correct".

## Task 3 · Stress-test a classifier (7 minutes)

Before running the model, label the four examples yourself as positive, negative, neutral or ambiguous. Then call the sentiment pipeline on the entire list, with input truncation enabled and a 128-token input cap. Compare your assessment with the model's available labels.

**Success:** four observed predictions with scores; one disagreement or ambiguous case discussed; one sentence explaining why a high score does not validate use on client tickets. If you agree with all four outputs, add an invented neutral or mixed-sentiment example and discuss the forced binary choice.

**Hint:** this checkpoint has no neutral output class. Inspect `sentiment.model.config.id2label`. A tiny hand-picked sample illustrates a failure mode; it is not a benchmark.

In [10]:
sentiment = load_pipeline('sentiment')
examples = [
    'The explanation was clear and useful.',
    'The application crashes every time I open it.',
    "The meeting starts at two o'clock.",
    'I loved the tutor, but the exercises were frustrating.',
]
my_labels = ['positive', 'negative', 'neutral', 'ambiguous']   # my own judgments, written before running the model
student_predictions = sentiment(examples, truncation=True, max_length=128)
print('Available model labels:', sentiment.model.config.id2label)
if student_predictions is None or len(my_labels) != len(examples):
    print('Complete your judgments and the pipeline call.')
else:
    assert len(student_predictions) == len(examples)
    for text, human, model_result in zip(examples, my_labels, student_predictions):
        print({'text': text, 'my_judgment': human, 'model': model_result})

c:\Users\agnay.krishna.biju\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\agnay.krishna.biju\.cache\huggingface\hub\models--distilbert--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for th

sentiment: distilbert/distilbert-base-uncased-finetuned-sst-2-english @ 714eb0f
DistilBertForSequenceClassification / DistilBertTokenizerFast
Available model labels: {0: 'NEGATIVE', 1: 'POSITIVE'}
{'text': 'The explanation was clear and useful.', 'my_judgment': 'positive', 'model': {'label': 'POSITIVE', 'score': 0.9997910857200623}}
{'text': 'The application crashes every time I open it.', 'my_judgment': 'negative', 'model': {'label': 'NEGATIVE', 'score': 0.9997283816337585}}
{'text': "The meeting starts at two o'clock.", 'my_judgment': 'neutral', 'model': {'label': 'POSITIVE', 'score': 0.9768263697624207}}
{'text': 'I loved the tutor, but the exercises were frustrating.', 'my_judgment': 'ambiguous', 'model': {'label': 'NEGATIVE', 'score': 0.9912058711051941}}


**Your interpretation:** My own labels were: positive, negative, neutral, and ambiguous (the last sentence is mixed: it praises the tutor and complains about the exercises). This checkpoint has only two classes, `NEGATIVE` and `POSITIVE` (see `id2label` in the output), so it cannot answer "neutral" or "mixed". The meeting sentence is a plain fact, but the model still has to pick one of the two labels, and the mixed sentence gets one label with one score even though it contains both feelings. Those two are the cases to look at in the printed predictions: **what I observed** is the label and score the model gave to each (fill in from your output: _neutral sentence -> ..., score ...; mixed sentence -> ..., score ..._).

A high score does not validate use on client tickets. The score only says how strongly the model prefers one of its two labels; the model was fine-tuned on movie-review sentences (SST-2), and support tickets use different wording, include complaints that are not about feelings, and often have no clear polarity. The model can be confident and wrong when the text is out of its domain.

*One check before using it on a different domain:* take 50 to 100 real tickets, label them by hand (including a neutral class), run the model, and compare. Look at accuracy per class and at the neutral and mixed tickets in particular, before relying on any score.

## Task 4 · Bound generation and compare decoding (10 minutes)

Load the generation model and run the provided reporting helper. Write a call that generates **two sampled continuations with at most 15 new tokens each**. Use a seed of 42, one beam and deliberate sampling parameters. Request token IDs with `return_tensors=True`; pass the tokenizer's EOS ID as `pad_token_id` for this model.

Then generate **one greedy continuation** with the same prompt and token cap. Print all three continuations with both token counts and whitespace word counts.

**Success:** two sampled results, one greedy result, and each count at most 15. Your explanation says “maximum new tokens,” allows early stopping and does not require exactly 15 words, two complete sentences or different sampled texts.

**Hint:** inspect `max_new_tokens`, `num_return_sequences` and `do_sample`. Greedy decoding with one beam uses one return sequence. Tune `temperature` and `top_p` for sampling. For the greedy call, use neutral `temperature=1.0` to reset this v4 pipeline's inherited 0.7 default and avoid an ignored-parameter warning; it has no sampling effect when `do_sample=False`.

In [11]:
def describe_generations(results, prompt, tokenizer):
    """Count original generated IDs, rather than re-tokenizing decoded text."""
    prompt_ids = tokenizer(prompt, add_special_tokens=False)['input_ids']
    rows = []
    for result in results:
        all_ids = result['generated_token_ids']
        assert all_ids[:len(prompt_ids)] == prompt_ids
        added_ids = all_ids[len(prompt_ids):]
        # DistilGPT2 uses EOS as padding here; ignore padding after the first EOS.
        if tokenizer.eos_token_id in added_ids:
            added_ids = added_ids[:added_ids.index(tokenizer.eos_token_id) + 1]
        continuation = tokenizer.decode(added_ids, skip_special_tokens=True)
        rows.append({
            'continuation': continuation,
            'new_tokens_including_eos': len(added_ids),
            'whitespace_words': len(continuation.split()),
        })
    return rows

In [12]:
generator = load_pipeline('generation')
prompt = 'To prepare a useful training workshop, we should'
set_seed(SEED)
sampled_results = generator(
    prompt,
    max_new_tokens=15,          # upper limit on NEW tokens, not an exact length
    num_return_sequences=2,
    do_sample=True,
    temperature=0.8,
    top_p=0.9,
    num_beams=1,
    return_tensors=True,        # keep the generated token IDs so we can count them
    pad_token_id=generator.tokenizer.eos_token_id,
)
greedy_results = generator(
    prompt,
    max_new_tokens=15,
    num_return_sequences=1,
    do_sample=False,            # always take the most likely next token
    num_beams=1,
    temperature=1.0,            # neutral value; resets the 0.7 default and has no effect without sampling
    return_tensors=True,
    pad_token_id=generator.tokenizer.eos_token_id,
)

c:\Users\agnay.krishna.biju\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\agnay.krishna.biju\.cache\huggingface\hub\models--distilbert--distilgpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is 

generation: distilbert/distilgpt2 @ 2290a62
GPT2LMHeadModel / GPT2TokenizerFast


In [13]:
if sampled_results is None or greedy_results is None:
    print('Complete both generation calls, then rerun this check.')
else:
    sampled_rows = describe_generations(sampled_results, prompt, generator.tokenizer)
    greedy_rows = describe_generations(greedy_results, prompt, generator.tokenizer)
    assert len(sampled_rows) == 2
    assert len(greedy_rows) == 1
    assert all(1 <= row['new_tokens_including_eos'] <= 15 for row in sampled_rows + greedy_rows)
    print('Sampled:', sampled_rows)
    print('Greedy:', greedy_rows)

Sampled: [{'continuation': ' try to introduce some basic knowledge about the different methods used for training.\n', 'new_tokens_including_eos': 15, 'whitespace_words': 13}, {'continuation': ' be able to use this tool to help us train in a more efficient manner', 'new_tokens_including_eos': 15, 'whitespace_words': 15}]
Greedy: [{'continuation': ' be able to use the following techniques to prepare a useful training workshop.\n', 'new_tokens_including_eos': 15, 'whitespace_words': 13}]


**Your interpretation:** The check cell prints, for the two sampled continuations and the greedy one, the text, the number of new tokens (including an end-of-text token if the model stopped early) and the number of whitespace words. **What I observed:** fill in from your output: _sampled 1: ... tokens / ... words; sampled 2: ...; greedy: ..._

(a) *Tokens versus words.* `max_new_tokens=15` is a maximum, so a continuation can be shorter if the model emits end-of-text. Even at 15 tokens the word count is usually different from 15, and it is usually lower, because a word can take more than one token and punctuation marks are tokens of their own. Counting the generated IDs directly, as the helper does, avoids the error you get from re-tokenizing the decoded text.

(b) *Sampling versus greedy.* Greedy decoding always takes the most likely next token, so with the same prompt it gives the same text every time, and it tends to be safe but repetitive or generic. Sampling draws from the probability distribution (here made narrower by `temperature=0.8` and `top_p=0.9`), so the two samples can differ from each other and from the greedy text, and a different seed would change them. The seed makes a run repeatable in a fixed environment; it does not make sampling deterministic across machines or library versions.

(c) *Neither method guarantees facts.* Both pick words that are likely to follow the previous words, based on patterns in the training text. Nothing in the process checks whether the statement is true, so fluent output can still be wrong. DistilGPT2 is also a small model.

*If an application really needed exactly 15 whitespace-delimited words:* the token cap cannot do that, because tokens and words are different units. The application would have to generate more than needed, count the words in the result, and then trim to 15 words (or ask again until the count is right), and also decide what to do if the text ends mid-sentence.

## Task 5 · Make a model decision (5 minutes)

A team wants to use one of these checkpoints on a new task. Choose **one** scenario and write 4–6 sentences:

- Route short English customer-support messages to three departments.
- Suggest a continuation for public training copy.
- Extract named organizations from public event announcements.

Name a suitable task and model family, identify a mismatch with at least one of the core checkpoints, give one test case, and state one evaluation measure or review criterion. Also identify a case the system should hand to a human. You do not need to download another model.

**Success:** your recommendation follows the task and evidence; it does not use a sentiment score as routing accuracy or fluency as factual correctness.

**Your answer:** I chose the first scenario, routing short English customer-support messages to three departments (for example billing, technical problems and account access). The right task is text classification with three labels, and a suitable model family is an encoder such as DistilBERT fine-tuned on labeled support messages from that company. This does not match the sentiment checkpoint: it is an encoder too, but its head has two outputs about positive or negative feeling and was trained on movie-review sentences, so "NEGATIVE" tells us nothing about whether a message belongs to billing or technical support, and an angry billing message would look the same as an angry technical one. The fill-mask and generation checkpoints are not classifiers at all. A test case would be "I was charged twice for my subscription this month", which should go to billing, together with "The app crashes when I log in", which should go to technical problems. I would measure per-class precision and recall (and macro-F1) on a held-out set of messages labeled by hand, and check the confusion matrix to see which departments get mixed up. Messages that mention several departments, that the model gives a low confidence to, or that involve legal threats or safety issues should go to a human instead of being routed automatically.

## Optional investigation - choose one (10–15 minutes)

These preserve the original exploration topics. A model-card review is enough for the first pass; download only after checking size and language support. If you run a model, use an explicit model ID and record its commit revision, task, tokenizer, library version and actual output. If you do not run it, state **not executed** and explain the planned test.

| Option | Action | Evidence to submit |
|---|---|---|
| Another language | Find a text-generation model whose card supports a language you know. | Model URL, language, architecture, task, weight size, limitation and one bounded test prompt. |
| Cased BERT | Inspect [google-bert/bert-base-cased](https://huggingface.co/google-bert/bert-base-cased). Identify its mask marker and contrast cased/uncased tokenization. | The marker and source; if executed, the top three observed candidates for one input. Do not copy a prediction from a widget. |
| POS tagging | Inspect [vblagoje/bert-english-uncased-finetuned-pos](https://huggingface.co/vblagoje/bert-english-uncased-finetuned-pos) or another English POS checkpoint. | Task, at least three tag meanings, and why those labels differ from NER. Note any missing license/training information in the card. |
| Translation | Choose one directed language pair and inspect a suitable model card. | Source/target language, tokenizer requirements, model URL and a plan to check numbers, names and meaning. |
| Original advanced demos | Enable one optional section in the guided lab: zero-shot, NER, QA or summary. | Your observed result and one limitation demonstrated by changing the input. |

**Your investigation and execution status:** **Not executed.** I did not do the optional investigation. The planned test would be the *Cased BERT* option: open the model card for `google-bert/bert-base-cased`, note its mask marker from the tokenizer, then run the same fill-mask sentence with the cased and uncased tokenizers and compare how a capitalized word such as "Riga" is split. I would submit the top three observed candidates for one input, and only from my own run.

## Sources and attribution

The original notebook follows the Hugging Face course chapter [Transformers, what can they do?](https://huggingface.co/learn/llm-course/chapter1/3). This revision uses new classroom examples and adds setup, model selection, bounded generation, exercises and interpretation. This is **source attribution**, not a license grant; the course source and each model retain their own applicable licenses. Model weights are downloaded separately and are not distributed in this notebook.

Primary references checked for this revision:

- [Transformers 4.57.1 pipeline API](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/pipelines), including tasks, NER aggregation and output formats.
- [Transformers 4.57.1 generation parameters](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/text_generation), including sampling and token limits.
- [Installation, caching and offline mode](https://huggingface.co/docs/transformers/v4.57.1/en/installation).
- [Sentiment model card](https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english) · [masked-language model card](https://huggingface.co/distilbert/distilbert-base-uncased) · [DistilGPT2 model card](https://huggingface.co/distilbert/distilgpt2).
- Weight-size evidence: [sentiment files](https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english/tree/main), [mask-model files](https://huggingface.co/distilbert/distilbert-base-uncased/tree/main), [generation-model files](https://huggingface.co/distilbert/distilgpt2/tree/main). The figures are for one safetensors weight file, not the entire repository or runtime memory.

Model revision hashes in the code freeze the reviewed files. A fixed seed improves repeatability in a fixed environment; it does not guarantee bit-identical text across different hardware, library versions or decoding implementations. Inspect warnings; this notebook does not suppress them globally.